# Turnwise decision-model smoke run

This Google Colab notebook checks the mechanics of loading, training for three steps, saving, and reloading a decision model. It uses only the pinned eight-row training export. The results are not a model-quality evaluation; do not use final-test data here.

Use a GPU runtime such as a Tesla T4. Upload `data/training/exports/initial-training.jsonl` to the Drive folder configured below before running the cells in order. The installation cell uses floating Unsloth dependencies, so record the actual versions for any new run.

Adapted from [Unsloth's Qwen3.5 decision-model notebook](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_5_%284B%29-Decision.ipynb), licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme). Modified for Turnwise's training-only smoke run.

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Load the pinned model

The T4 runtime used for the smoke run fell back to float32 for Qwen3.5. The freshly initialized decision head is used only for an option-compatibility check.

In [ ]:
import torch
from unsloth import FastDecisionModel

assert torch.cuda.is_available(), "Colab has no active GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name="unsloth/Qwen3.5-0.8B",
    revision="23c69c53358a07516b5827588b3fdb12ae78fd65",
    max_seq_length=2048,
    load_in_4bit=True,
)

free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free VRAM: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

work_dir = Path("/content/drive/MyDrive/TurnwiseSmoke")
data_path = work_dir / "initial-training.jsonl"

In [ ]:
import hashlib
import json

expected_hash = "58b4d8e6fc8f43dccb551a2276988ba5871eeee097dce74772d0ea0f9fedd415"
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == expected_hash, "Unexpected training file"

rows = [json.loads(line) for line in data_path.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) == 8, f"Expected 8 rows, found {len(rows)}"
assert all(set(row) == {"state", "questions", "gold"} for row in rows)
print(f"Loaded {len(rows)} verified training rows")

In [ ]:
from unsloth import FastDecisionModel

row = rows[0]

FastDecisionModel.for_inference(model)
answers = FastDecisionModel.predict(model, tokenizer, row["state"], row["questions"])

assert set(answers) == set(row["questions"])
for question_id, result in answers.items():
    answer = result["answer"]
    assert answer in row["questions"][question_id]["criteria"], question_id
    print(f"{question_id}: {answer}")

print("All six answers are valid original options.")

In [ ]:
train_items, build_report = FastDecisionModel.build_dataset(rows, tokenizer, model)

print("Build report:", build_report)
print("Prepared items:", len(train_items))

assert len(train_items) == 8, "Expected one prepared item per training row"
assert build_report.get("total") == 48, "Expected 48 decisions"
assert build_report.get("skipped") == 0, "Some decisions were skipped"
assert build_report.get("truncated") == 0, "Some inputs were truncated"

In [ ]:
import torch

assert not any("lora_" in name for name, _ in model.named_parameters()), (
    "LoRA parameters already exist; do not attach adapters twice."
)

model = FastDecisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = [
    (name, parameter.numel())
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("Trainable parameters:", sum(count for _, count in trainable))
print("Trainable LoRA tensors:", sum("lora_" in name for name, _ in trainable))
print("Trainable head tensors:", [name for name, _ in trainable if "head" in name])

free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free VRAM: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")

### Run three training steps

This is a mechanics check, not a held-out evaluation. No validation, calibration, or final-test rows are used.

In [ ]:
from unsloth import DecisionTrainer
from transformers import TrainingArguments

assert len(rows) == 8
assert len(train_items) == 8

training_args = TrainingArguments(
    output_dir=str(work_dir / "output"),
    max_steps=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    warmup_steps=0,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    seed=3407,
    data_seed=3407,
    fp16=False,
    bf16=False,
)

trainer = DecisionTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_items,
    args=training_args,
    head_learning_rate=1e-4,
)

train_result = trainer.train()

print("Global step:", trainer.state.global_step)
print("Training metrics:", train_result.metrics)
assert trainer.state.global_step == 3, "Training did not complete exactly three steps"

### Save and verify the checkpoint

Save the adapter, decision head, and tokenizer in a new Drive folder, then reload them and verify the saved weights and option keys.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import hashlib

assert trainer.state.global_step == 3

checkpoint_dir = (
    work_dir / "checkpoints"
    / f"smoke-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex[:8]}"
)
checkpoint_dir.mkdir(parents=True, exist_ok=False)

model.save_pretrained(str(checkpoint_dir))
tokenizer.save_pretrained(str(checkpoint_dir))

print("Checkpoint:", checkpoint_dir)
for path in sorted(checkpoint_dir.rglob("*")):
    if path.is_file():
        with path.open("rb") as file:
            digest = hashlib.file_digest(file, "sha256").hexdigest()
        print(f"{path.relative_to(checkpoint_dir)}  {path.stat().st_size} bytes  SHA256: {digest}")

In [ ]:
import json

decision_config = json.loads(
    (checkpoint_dir / "unsloth_decision_config.json").read_text(encoding="utf-8")
)
adapter_config = json.loads(
    (checkpoint_dir / "adapter_config.json").read_text(encoding="utf-8")
)
expected_model = "unsloth/Qwen3.5-0.8B"
expected_revision = "23c69c53358a07516b5827588b3fdb12ae78fd65"

assert decision_config.get("base_model") == expected_model
assert decision_config.get("base_revision") == expected_revision
assert adapter_config.get("base_model_name_or_path") == expected_model
assert adapter_config.get("revision") == expected_revision
assert getattr(model.encoder.config, "_commit_hash", None) == expected_revision
print("Checkpoint and loaded encoder report the expected model revision.")

In [ ]:
import gc
import torch
from safetensors.torch import load_file
from unsloth import FastDecisionModel

del trainer, model, tokenizer
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=str(checkpoint_dir),
    max_seq_length=2048,
    load_in_4bit=True,
)

reloaded_commit = getattr(model.encoder.config, "_commit_hash", None)
print("Reloaded encoder commit:", reloaded_commit)
assert reloaded_commit == expected_revision, "Reloaded encoder revision differs"
assert any("lora_" in name for name, _ in model.named_parameters()), "LoRA not found"

saved_head = load_file(str(checkpoint_dir / "joint_head.safetensors"))
loaded_head = model.head.state_dict()
assert saved_head.keys() == loaded_head.keys(), "Head tensor names differ"

mismatched = [
    name for name, saved in saved_head.items()
    if not torch.equal(loaded_head[name].detach().cpu().to(saved.dtype), saved)
]
assert not mismatched, f"Head weights differ: {mismatched}"
print("Saved head weights match the reloaded head.")

FastDecisionModel.for_inference(model)
row = rows[0]
answers = FastDecisionModel.predict(model, tokenizer, row["state"], row["questions"])

assert set(answers) == set(row["questions"])
for question_id, result in answers.items():
    assert result["answer"] in row["questions"][question_id]["criteria"], question_id

print("All six reloaded-model answers are valid original options.")

In [ ]:
from peft import get_peft_model_state_dict
from safetensors.torch import load_file
import torch

saved_adapter = load_file(str(checkpoint_dir / "adapter_model.safetensors"))
loaded_adapter = get_peft_model_state_dict(model.encoder)

missing = sorted(saved_adapter.keys() - loaded_adapter.keys())
extra = sorted(loaded_adapter.keys() - saved_adapter.keys())
print("Missing tensor names:", missing[:5], "count:", len(missing))
print("Extra tensor names:", extra[:5], "count:", len(extra))

assert not missing and not extra, "Adapter tensor names differ"

mismatched = [
    name for name, saved in saved_adapter.items()
    if not torch.equal(loaded_adapter[name].detach().cpu().to(saved.dtype), saved)
]
print("Mismatched tensors:", mismatched[:5], "count:", len(mismatched))
assert not mismatched, "Adapter weights differ"

print(f"All {len(saved_adapter)} saved adapter tensors match.")